# Reference copy - needs non-public data

이 노트북은 원본 연구 노트북의 **참고용 사본**입니다. 실행에 필요한 02 노트북의 `portfolio.csv`와 월간 팩터 파일(`factors_monthly.csv`)은(는) 유료/비공개 데이터라 이 저장소에 포함하지 않았으므로 **그대로는 끝까지 실행되지 않습니다.** 출력, 실행 카운트, 로컬 경로는 제거했습니다. 실행 가능한 데모는 `00_public_sample_workflow.ipynb`(합성 데이터)를 사용하세요.

원본은 학회 세션 자료(단계별 폴더)였고, 파일 경로는 `./input/`, `./output/` 기준의 중립 경로로 바꿨습니다.

---
### **Performance evalution**

포트폴리오 성과 평가
- CAGR
- Volatility
- Sharpe Ratio
- MDD
- Turnover
- 벤치마크 비교

---
##### **데이터 로드**

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Investment Factor 계산에 필요한 데이터를 input 폴더에서 불러옴
portfolio = pd.read_csv("./input/portfolio.csv", index_col=0, parse_dates=True)         # 포트폴리오
factors   = pd.read_csv("./input/factors_monthly.csv", index_col=0, parse_dates=True)   # 팩터(MKT, SMB, HML, MOM, RF) (독립변수)

In [ ]:
portfolio.tail()

---
### **1. 수익률 분석**

##### Code

In [ ]:
# 1. 두 데이터 공통 구간 맞추기
df = pd.concat([portfolio['Return'], factors], axis=1, join="inner").dropna()

In [ ]:
# 2. 종속변수: 포트폴리오 초과수익률
y = df['Return'] - df['RF']

# 3. 독립변수: MKT, SMB, HML, MOM
X = pd.DataFrame({
    "MKT": df['KOSPI'] - df['RF'],
    "SMB": df['SMB'],
    "HML": df['HML'],
    "MOM": df['MOM']
}, index=df.index)

X = sm.add_constant(X, has_constant='add')

# 4. OLS 회귀
model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":12})  # Newey-West 표준오차

##### Output

In [ ]:
print(model.summary())

---
### **2. 성과지표 계산**

##### Code

함수 정의

In [ ]:
def performance_metrics(portfolio):

    portfolio_NAV    = portfolio['NAV'][1:]
    portfolio_return = portfolio['Return'][1:]
    total_trade      = portfolio['Trade'][1:]
    initial_NAV      = portfolio['NAV'].iloc[0]

    # 1. 연평균 수익률 (CAGR)
    total_months = len(portfolio_return)
    years = total_months / 12
    final_value = portfolio_NAV.iloc[-1]
    CAGR = (final_value / initial_NAV) ** (1 / years) - 1

    # 2. 월간 변동성 (Annualized Volatility)
    vol_monthly = portfolio_return.std()
    vol_annual = vol_monthly * np.sqrt(12)

    # 3. Sharpe Ratio (Rf=0 가정)
    mean_return_monthly = portfolio_return.mean()
    sharpe_ratio = (mean_return_monthly * 12) / vol_annual

    # 4. 최대 낙폭 (MDD)
    cummax_NAV = portfolio_NAV.cummax()
    drawdown = portfolio_NAV / cummax_NAV - 1
    MDD = drawdown.min()

    # 5. 월간 Turnover & 평균 Turnover
    monthly_turnover = total_trade / portfolio_NAV  # NAV 대비 총 거래액
    avg_turnover = monthly_turnover.mean()

    return {
        "CAGR": CAGR,
        "Volatility (ann.)": vol_annual,
        "Sharpe Ratio": sharpe_ratio,
        "MDD": MDD,
        "Average Turnover (monthly)": avg_turnover,
        "Turnover Series": monthly_turnover
    }

In [ ]:
metrics = performance_metrics(portfolio)

##### Output

In [ ]:
for k, v in metrics.items():
    if k != "Turnover Series":  # Turnover Series는 별도 DataFrame으로 보는 게 좋음
        print(f"{k}: {v:.4f}")

---
### **3. Plot**

---
**NAV Plot**

##### Code

In [ ]:
df_ret = pd.concat([portfolio['Return'].dropna(), factors['KOSPI']], axis=1, join='inner')
df_ret.columns = ['Portfolio', 'KOSPI']
df_ret.index.name = 'Date'

In [ ]:
df_nav = (1 + df_ret).cumprod()
df_nav.columns = ['Portfolio', 'KOSPI']

##### Plot

In [ ]:
df_nav.plot()

---
**로그누적수익률 Plot**

In [ ]:
np.log1p(df_ret).cumsum().plot()